# Lesson 23 Lab — Paged KV Cache Addressing

**Puzzle:** When block tables, logical tokens, physical blocks, and gather layout change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates block tables, logical tokens, physical blocks, and gather layout and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Paged attention separates a token's logical position from the physical KV block. A block table lookup selects the physical block, while modulo arithmetic selects the slot. The kernel must preserve this indirection without losing width-wise coalescing.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["block tables, logical tokens, physical blocks, and gather layout"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Correct contiguous-cache tests do not cover block-table permutations, partial final blocks, or invalid slot ownership.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 23
LESSON_TITLE = 'Paged KV Cache Addressing'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260836
}


## 5. Freeze the experiment

**Experiment:** Implement a Triton paged gather over a randomized logical-to-physical block table and compare advanced indexing.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.017103999853134155,
  "secondary": 0.03899199888110161,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "block_table_entries": 128,
    "triton_samples_ms": [
      0.03001599945127964,
      0.02160000056028366,
      0.01942400075495243,
      0.019872000440955162,
      0.018271999433636665,
      0.020031999796628952,
      0.01775999926030636,
      0.01711999997496605,
      0.016416000202298164,
      0.016287999227643013,
      0.01708799973130226,
      0.016736000776290894,
      0.017376000061631203,
      0.021183999255299568,
      0.016224000602960587,
      0.015936000272631645,
      0.01648000068962574,
      0.016127999871969223,
      0.016672000288963318,
      0.016736000776290894
    ],
    "pytorch_samples_ms": [
      0.041280001401901245,
      0.04032000154256821,
      0.03964800015091896,
      0.0390079990029335,
      0.0395519994199276,
      0.040511999279260635,
      0.0395519994199276,
      0.039423998445272446,
      0.

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton gather median | 0.0171 ms |
| PyTorch indexing median | 0.0390 ms |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The Triton paged gather followed 128 logical-to-physical block entries in 0.0171 ms and matched advanced indexing exactly.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Validate address mapping independently before embedding the gather inside an attention reduction.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 23,
  "title": "Paged KV Cache Addressing",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260836
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.017103999853134155,
    "secondary": 0.03899199888110161,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "block_table_entries": 128,
      "triton_samples_ms": [
        0.03001599945127964,
        0.02160000056028366,
        0.01942400075495243,
        0.019872000440955162,
        0.018271999433636665,
        0.020031999796628952,
        0.01775999926030636,
        0.01711999997496605,
        0.016416000202298164,
        0.016287999227643013,
        0.01708799973130226,
        0.016736000776290894,
        0.017376000061631203,
        0

## 10. Make the bounded decision

> Validate address mapping independently before embedding the gather inside an attention reduction.

**Failure analysis:** Correct contiguous-cache tests do not cover block-table permutations, partial final blocks, or invalid slot ownership.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
